[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vpasumarthi/aimd-tutorial/blob/main/notebooks/02_water_on_pt111.ipynb)

# Water on Pt(111): Surface-Water Interface

This tutorial covers AIMD simulation of water on a Pt(111) surface -- a model system for electrocatalysis and surface chemistry. We use a 2x2 slab (3 Pt layers) with ~10 H$_2$O molecules.

**Learning objectives:**
- Build a metal-water slab model with pymatgen
- Configure VASP for metal-water AIMD (smearing, dipole corrections)
- Compute the water density profile normal to the surface
- Analyze water molecule orientations near the Pt surface

In [ ]:
# --- Run this cell first in Google Colab ---
import sys
if 'google.colab' in sys.modules:
    %pip install -q ase pymatgen nglview scipy
    from google.colab import output
    output.enable_custom_widget_manager()

## Background

The structure of water at metal surfaces governs reaction mechanisms in electrocatalysis, corrosion, and fuel cells. On Pt(111), the first water layer forms an ordered ice-like bilayer at low coverage, but under AIMD conditions at finite temperature, the structure is dynamic and disordered.

Key structural questions:
- How does the water density vary with distance from the surface?
- Do water molecules preferentially orient their dipole toward or away from the metal?
- How thick is the interfacial region before bulk-like behavior is recovered?

Our model: a 2x2 Pt(111) slab with 3 layers (12 Pt atoms), ~10 H$_2$O molecules, and ~15 Angstrom of vacuum. The bottom Pt layer is fixed during MD to anchor the slab.

## Structure Setup with pymatgen

We use pymatgen's `SlabGenerator` to create the Pt(111) surface, then add water molecules above it.

In [ ]:
from pymatgen.core import Structure, Lattice
from pymatgen.core.surface import SlabGenerator
from pymatgen.io.vasp import Poscar

# Pt FCC bulk (PBE lattice constant)
a_pt = 3.92  # Angstrom
pt_bulk = Structure(
    Lattice.cubic(a_pt),
    ['Pt'] * 4,
    [[0, 0, 0], [0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]],
)

# Generate Pt(111) slab: 3 layers, 2x2 supercell, ~15 A vacuum
slab_gen = SlabGenerator(
    pt_bulk,
    miller_index=(1, 1, 1),
    min_slab_size=6.0,     # minimum slab thickness in Angstrom
    min_vacuum_size=15.0,  # vacuum thickness in Angstrom
    center_slab=True,
)

slab = slab_gen.get_slabs()[0]
# Make 2x2 supercell in the surface plane
slab.make_supercell([2, 2, 1])

print(f'Slab: {len(slab)} atoms')
print(f'Cell: {slab.lattice.abc}')
print(f'Species: {dict(slab.composition.as_dict())}')

# Write POSCAR for inspection
# Poscar(slab).write_file('POSCAR_slab')

In [ ]:
import nglview as nv
from pymatgen.io.ase import AseAtomsAdaptor

# Interactive 3D view of the bare Pt(111) slab
slab_ase = AseAtomsAdaptor.get_atoms(slab)
view = nv.show_ase(slab_ase)
view.add_unitcell()
view

**Adding water:** Place ~10 H$_2$O molecules above the slab. For a well-equilibrated starting point, use Packmol to fill the vacuum region with water at the target density, then relax with a classical force field before AIMD. The final structure should look like:

```
Water on Pt(111) - 2x2x3 slab + 10 H2O
1.0
  5.5400  0.0000  0.0000
 -2.7700  4.7975  0.0000
  0.0000  0.0000 25.0000
Pt   O    H
12   10   20
Selective dynamics
Cartesian
  ... (Pt positions -- bottom layer with F F F, top layers with T T T)
  ... (O and H positions with T T T)
```

Use `Selective dynamics` to freeze the bottom Pt layer (`F F F`) while allowing the top layers and all water to move (`T T T`).

## VASP Input Files

### INCAR

```
SYSTEM = Water on Pt(111) - 2x2x3 + 10 H2O NVT

# Electronic structure
PREC    = Normal
ENCUT   = 400
ALGO    = VeryFast
EDIFF   = 1E-5
ISMEAR  = 1          # Methfessel-Paxton (metallic system)
SIGMA   = 0.2
LREAL   = Auto
NELMIN  = 4

# Molecular dynamics
IBRION  = 0
MDALGO  = 2          # Nose-Hoover thermostat
SMASS   = 1.0
POTIM   = 1.0
NSW     = 5000
TEBEG   = 400
TEEND   = 400
ISIF    = 2

# Dipole correction (essential for asymmetric slab)
LDIPOL  = .TRUE.
IDIPOL  = 3          # Correct along z (surface normal)

# Output
NWRITE  = 0
NBLOCK  = 1
LWAVE   = .FALSE.
LCHARG  = .FALSE.
```

**Key differences from bulk water:**
- `ISMEAR = 1` with `SIGMA = 0.2`: Methfessel-Paxton smearing for the metallic Pt states. Gaussian smearing (`ISMEAR = 0`) would also work but MP is standard for metals.
- `LDIPOL = .TRUE.` / `IDIPOL = 3`: Corrects the artificial dipole interaction between periodic slab images. Essential when water is on only one side of the slab.
- Selective dynamics in POSCAR to freeze the bottom Pt layer.

### KPOINTS

The smaller surface cell requires denser k-point sampling than the bulk water box.

```
Automatic mesh
0
Gamma
2 2 1
0 0 0
```

Only 1 k-point along z since the slab is non-periodic in that direction (vacuum separates images).

### POTCAR

```bash
cat $VASP_PP_PATH/Pt/POTCAR $VASP_PP_PATH/O/POTCAR $VASP_PP_PATH/H/POTCAR > POTCAR
```

Use the standard `Pt` PAW potential (10 valence electrons). Match the species order in POSCAR.

---
## Trajectory Analysis

Copy `XDATCAR` and `OSZICAR` from your completed run into `data/water_pt111/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read

plt.rcParams.update({
    'figure.figsize': (6, 4),
    'font.size': 12,
    'axes.linewidth': 1.2,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
})

In [ ]:
trajectory = read('data/water_pt111/XDATCAR', index=':')

n_frames = len(trajectory)
timestep = 1.0  # fs
equil_steps = 500  # frames to discard

print(f'Loaded {n_frames} frames, {len(trajectory[0])} atoms')
print(f'Total simulation time: {n_frames * timestep / 1000:.1f} ps')

In [ ]:
import nglview as nv
from IPython.display import display

# Interactive 3D view of the slab + water system
view = nv.show_ase(trajectory[0])
view.add_unitcell()
display(view)

# Trajectory playback -- use the slider to scrub through frames
traj_view = nv.show_asetraj(trajectory[::10])
traj_view.add_unitcell()
traj_view

### Water Density Profile

The water density profile $\rho(z)$ shows how water molecules distribute themselves above the Pt surface. Peaks in $\rho(z)$ indicate adsorption layers.

We compute the profile by binning oxygen atom $z$-coordinates across all frames, then normalizing to get a number density.

In [ ]:
def compute_density_profile(trajectory, element='O', n_bins=100,
                            start_frame=0, reference_element='Pt'):
    """Compute atomic density profile along z.

    The z-origin is set to the mean position of the topmost surface
    layer atoms (reference_element) so that the profile is relative
    to the surface.
    """
    frames = trajectory[start_frame:]
    symbols = np.array(frames[0].get_chemical_symbols())
    idx_target = np.where(symbols == element)[0]
    idx_ref = np.where(symbols == reference_element)[0]
    cell_z = frames[0].get_cell()[2, 2]

    all_z = []
    for atoms in frames:
        pos = atoms.get_positions()
        # Surface reference: top Pt layer (highest mean z among Pt atoms)
        ref_z = np.max(pos[idx_ref, 2])
        z_shifted = pos[idx_target, 2] - ref_z
        all_z.extend(z_shifted)

    all_z = np.array(all_z)

    # Bin into histogram
    z_min, z_max = 0, 12  # Angstrom above the top Pt layer
    hist, bin_edges = np.histogram(all_z, bins=n_bins, range=(z_min, z_max))
    z_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    dz = bin_edges[1] - bin_edges[0]

    # Convert to number density (atoms / A^3)
    # Area of the surface unit cell
    cell = frames[0].get_cell()
    area = np.linalg.norm(np.cross(cell[0], cell[1]))
    n_frames_used = len(frames)
    density = hist / (n_frames_used * area * dz)

    return z_centers, density


z_o, rho_o = compute_density_profile(trajectory, element='O',
                                      start_frame=equil_steps)
z_h, rho_h = compute_density_profile(trajectory, element='H',
                                      start_frame=equil_steps)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(z_o, rho_o, lw=1.5, color='C0', label='O')
ax.plot(z_h, rho_h, lw=1.5, color='C1', label='H')
ax.set_xlabel(r'$z - z_{\mathrm{Pt}}^{\mathrm{top}}$ ($\mathrm{\AA}$)')
ax.set_ylabel(r'Number density ($\mathrm{\AA}^{-3}$)')
ax.set_title('Water density profile above Pt(111)')
ax.legend()
ax.set_xlim(0, 12)
plt.tight_layout()
plt.show()

**Interpreting the profile:** A sharp first peak in the oxygen density at ~2.2-2.5 Angstrom above the Pt surface indicates the chemisorbed water layer. A second, broader peak at ~5-6 Angstrom shows the second solvation layer. The hydrogen profile reveals whether H atoms in the first layer point toward or away from the surface.

### Water Orientation Analysis

The orientation of water molecules near the surface reveals how the hydrogen-bond network reorganizes at the interface. We compute two angles for each water molecule:

1. **Dipole angle** $\theta_d$: angle between the water dipole (bisector of H-O-H) and the surface normal. $\theta_d = 0^\circ$ means the dipole points away from the surface (H atoms up); $\theta_d = 180^\circ$ means the dipole points toward the surface.
2. **OH tilt angle** $\theta_{OH}$: angle between each O-H bond vector and the surface normal.

In [ ]:
def identify_water_molecules(atoms, o_h_cutoff=1.3):
    """Group atoms into water molecules based on O-H distances.

    Returns list of (O_index, H1_index, H2_index) tuples.
    """
    symbols = np.array(atoms.get_chemical_symbols())
    o_indices = np.where(symbols == 'O')[0]
    h_indices = np.where(symbols == 'H')[0]

    molecules = []
    for o_idx in o_indices:
        dists = atoms.get_distances(o_idx, h_indices, mic=True)
        bonded_h = h_indices[np.argsort(dists)[:2]]
        # Verify both are within bonding distance
        if all(atoms.get_distance(o_idx, h, mic=True) < o_h_cutoff for h in bonded_h):
            molecules.append((o_idx, bonded_h[0], bonded_h[1]))
    return molecules


def compute_water_orientations(trajectory, z_min, z_max, start_frame=0,
                               reference_element='Pt'):
    """Compute dipole and OH angles for water molecules within a z-range.

    z_min, z_max define the slab of interest relative to the top Pt layer.
    """
    frames = trajectory[start_frame:]
    surface_normal = np.array([0, 0, 1])

    dipole_angles = []
    oh_angles = []

    for atoms in frames:
        symbols = np.array(atoms.get_chemical_symbols())
        pos = atoms.get_positions()
        idx_ref = np.where(symbols == reference_element)[0]
        z_surface = np.max(pos[idx_ref, 2])

        molecules = identify_water_molecules(atoms)
        for o_idx, h1_idx, h2_idx in molecules:
            z_o = pos[o_idx, 2] - z_surface
            if z_o < z_min or z_o > z_max:
                continue

            # O-H bond vectors (minimum image)
            r_oh1 = atoms.get_distance(o_idx, h1_idx, mic=True, vector=True)
            r_oh2 = atoms.get_distance(o_idx, h2_idx, mic=True, vector=True)

            # Dipole direction (bisector of H-O-H, pointing from O toward H's)
            dipole = r_oh1 + r_oh2
            dipole /= np.linalg.norm(dipole)
            cos_d = np.dot(dipole, surface_normal)
            dipole_angles.append(np.degrees(np.arccos(np.clip(cos_d, -1, 1))))

            # OH tilt angles
            for r_oh in [r_oh1, r_oh2]:
                r_oh_norm = r_oh / np.linalg.norm(r_oh)
                cos_oh = np.dot(r_oh_norm, surface_normal)
                oh_angles.append(np.degrees(np.arccos(np.clip(cos_oh, -1, 1))))

    return np.array(dipole_angles), np.array(oh_angles)

In [ ]:
# First layer: 0-3.5 A above surface
# Second layer / bulk-like: 3.5-8 A above surface
layers = [
    ('First layer (0-3.5 A)', 0, 3.5),
    ('Second layer (3.5-8 A)', 3.5, 8.0),
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for label, z_min, z_max in layers:
    dipole_ang, oh_ang = compute_water_orientations(
        trajectory, z_min, z_max, start_frame=equil_steps
    )
    if len(dipole_ang) == 0:
        print(f'No water molecules found in {label}')
        continue

    # Weight by 1/sin(theta) to correct for solid angle
    # (uniform distribution in cos(theta) appears uniform in the histogram)
    theta_bins = np.linspace(0, 180, 37)

    axes[0].hist(dipole_ang, bins=theta_bins, density=True,
                 alpha=0.6, label=label)
    axes[1].hist(oh_ang, bins=theta_bins, density=True,
                 alpha=0.6, label=label)

axes[0].set_xlabel(r'Dipole angle $\theta_d$ (deg)')
axes[0].set_ylabel('Probability density')
axes[0].set_title('Water dipole orientation')
axes[0].legend(fontsize=9)

axes[1].set_xlabel(r'O-H angle $\theta_{OH}$ (deg)')
axes[1].set_ylabel('Probability density')
axes[1].set_title('O-H bond orientation')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

**Interpreting orientations:**
- First-layer water on Pt(111) typically shows a bimodal dipole distribution: some molecules lie flat ($\theta_d \approx 90^\circ$) while others point one H toward the surface ($\theta_d > 90^\circ$) to form Pt-HO bonds.
- The second layer should approach a more uniform distribution, resembling bulk water.
- If most OH bonds point toward the surface ($\theta_{OH} > 90^\circ$), this indicates "H-down" water configurations.

### Layer-Resolved Pair Distribution

We can also examine how O-O correlations differ between the first adsorption layer and the bulk-like region. This reveals whether the surface imposes lateral ordering on the water network.

In [ ]:
def compute_lateral_rdf(trajectory, z_min, z_max, r_max=6.0, n_bins=100,
                        start_frame=0, reference_element='Pt'):
    """Compute in-plane O-O RDF for water oxygens within a z-slab.

    Uses the 2D (lateral) distance in the xy-plane.
    """
    frames = trajectory[start_frame:]
    r_edges = np.linspace(0, r_max, n_bins + 1)
    r_centers = 0.5 * (r_edges[:-1] + r_edges[1:])
    hist = np.zeros(n_bins)

    n_counted = 0
    for atoms in frames:
        symbols = np.array(atoms.get_chemical_symbols())
        pos = atoms.get_positions()
        idx_ref = np.where(symbols == reference_element)[0]
        z_surface = np.max(pos[idx_ref, 2])

        idx_o = np.where(symbols == 'O')[0]
        z_rel = pos[idx_o, 2] - z_surface
        in_layer = idx_o[(z_rel >= z_min) & (z_rel < z_max)]

        if len(in_layer) < 2:
            continue

        for i_idx, i in enumerate(in_layer):
            for j in in_layer[i_idx + 1:]:
                # Full 3D MIC distance for proper pair counting
                d = atoms.get_distance(i, j, mic=True)
                if d < r_max:
                    b = int(d / (r_max / n_bins))
                    if b < n_bins:
                        hist[b] += 2  # count (i,j) and (j,i)
        n_counted += len(in_layer)

    # Normalize as 3D RDF (approximate, since atoms are in a slab)
    n_frames_used = len(frames)
    volume = frames[0].get_volume()
    n_avg = n_counted / n_frames_used if n_frames_used > 0 else 1
    rho = n_avg / volume
    shell_volumes = (4.0 / 3.0) * np.pi * (r_edges[1:]**3 - r_edges[:-1]**3)

    if n_counted > 0:
        g_r = hist / (n_counted * rho * shell_volumes)
    else:
        g_r = np.zeros(n_bins)

    return r_centers, g_r


fig, ax = plt.subplots(figsize=(6, 4))

for label, z_min, z_max, color in [
    ('First layer', 0, 3.5, 'C0'),
    ('Second layer', 3.5, 8.0, 'C1'),
]:
    r, g = compute_lateral_rdf(trajectory, z_min, z_max,
                                start_frame=equil_steps)
    ax.plot(r, g, lw=1.5, color=color, label=label)

ax.axhline(1.0, ls='--', color='gray', lw=0.8)
ax.set_xlabel(r'$r$ ($\mathrm{\AA}$)')
ax.set_ylabel(r'$g_{\mathrm{OO}}(r)$')
ax.set_title('Layer-resolved O-O pair distribution')
ax.set_xlim(2, 6)
ax.legend()
plt.tight_layout()
plt.show()

## Summary

In this tutorial you learned to:
1. Build a Pt(111) slab model with pymatgen and add water
2. Configure VASP for metal-water AIMD (Methfessel-Paxton smearing, dipole corrections)
3. Compute water density profiles to identify adsorption layers
4. Analyze water dipole and O-H bond orientations as a function of distance from the surface
5. Compute layer-resolved pair distributions

**Next steps:**
- Increase the water layer thickness (20-30 H$_2$O) to better capture the transition to bulk-like behavior
- Compare with other metal surfaces (Au(111), Cu(111)) to see how water structure depends on the substrate
- Add an applied electric field (`EFIELD` in VASP) to model electrochemical conditions
- Compute the work function from the electrostatic potential profile